# RAIDS-NIDS v0.20 amended external guard study

This notebook preserves the failed v0.19 suite and implements the separately
numbered construction amendment. Run it from the `raids-nids` project root.

Do not change the family list, event gates, guard candidates, window
boundaries, or model seeds. Stop after the event review until those artifacts
have been checked.


In [1]:
from pathlib import Path
import hashlib
import itertools
import json

import pandas as pd
import raids_nids
import river

from raids_nids.audit import audit_dataset
from raids_nids.config import deep_merge, load_yaml
from raids_nids.guard_benchmark import (
    aggregate_guard_benchmarks,
    run_guard_benchmark,
)
from raids_nids.unsw_amendment import build_unsw_amended_event_suite

print("raids-nids:", raids_nids.__version__)
print("river:", river.__version__)
assert raids_nids.__version__ == "0.1.10"
assert river.__version__ == "0.25.0"


raids-nids: 0.1.10
river: 0.25.0


In [2]:
RAW = Path("data/raw/NF-UNSW-NB15-v3.csv")
CACHE = Path("data/derived/v019_unsw_temporal.npz")
CACHE_METADATA = CACHE.with_suffix(".json")
V019_SUITE = Path(
    "data/derived/v019_unsw_events/"
    "NF-UNSW-NB15-v3-v019-suite-manifest.json"
)
PROTOCOL = Path(
    "configs/protocols/v020_external_guard_amendment.yaml"
)
EVENT_DIR = Path("data/derived/v020_unsw_events")
RESULTS_DIR = Path("results/v020_external_guard_amendment/runs")
AGGREGATE_DIR = Path("results/v020_external_guard_amendment/aggregate")

EXPECTED_RAW_SHA256 = (
    "4ebb97bd74412d566137d95a6fc3ffd8f"
    "374f1cf8cfe204d007848e7a668f9b5"
)
EXPECTED_CACHE_SHA256 = (
    "215b2ea90aa5183c3cd99a20ba5d24c2"
    "5d1dbe35ebe0f1775ab2889b245f240a"
)
EXPECTED_PROTOCOL_SHA256 = (
    "04067670316a07ab87310879a7bd64689"
    "fe6d7e52758c7dc97d2b5409fe7402b"
)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


## 1. Verify the official dataset, cache, protocol, and failed v0.19 record

The v0.19 suite manifest is immutable evidence. This notebook stops if it is
missing or does not report the original three construction failures.


In [3]:
for required in [RAW, CACHE, CACHE_METADATA, V019_SUITE, PROTOCOL]:
    assert required.exists(), f"Required file is missing: {required.resolve()}"

input_hashes = {
    "raw_dataset": sha256(RAW),
    "temporal_cache": sha256(CACHE),
    "v019_suite_manifest": sha256(V019_SUITE),
    "v020_protocol": sha256(PROTOCOL),
}
assert input_hashes["raw_dataset"] == EXPECTED_RAW_SHA256
assert input_hashes["temporal_cache"] == EXPECTED_CACHE_SHA256
assert input_hashes["v020_protocol"] == EXPECTED_PROTOCOL_SHA256

v019_suite = json.loads(V019_SUITE.read_text(encoding="utf-8"))
assert (
    v019_suite["protocol_id"]
    == "RAIDS-NIDS-v0.19-external-guard-comparison"
)
assert v019_suite["constructed_count"] == 0
assert v019_suite["failed_count"] == 3
assert {
    row["family"] for row in v019_suite["outcomes"]
} == {"DoS", "Exploits", "Reconnaissance"}

print(
    json.dumps(
        {
            "input_hashes": input_hashes,
            "v019_constructed_count": v019_suite["constructed_count"],
            "v019_failed_count": v019_suite["failed_count"],
            "v019_families": [
                [row["family"], row["status"]]
                for row in v019_suite["outcomes"]
            ],
        },
        indent=2,
    )
)


{
  "input_hashes": {
    "raw_dataset": "4ebb97bd74412d566137d95a6fc3ffd8f374f1cf8cfe204d007848e7a668f9b5",
    "temporal_cache": "215b2ea90aa5183c3cd99a20ba5d24c25d1dbe35ebe0f1775ab2889b245f240a",
    "v019_suite_manifest": "18ed1aaf5926debdce53cf14343d69f1024fd387ff35f0055ee70a9b711cf0be",
    "v020_protocol": "04067670316a07ab87310879a7bd64689fe6d7e52758c7dc97d2b5409fe7402b"
  },
  "v019_constructed_count": 0,
  "v019_failed_count": 3,
  "v019_families": [
    [
      "DoS",
      "failed_event_construction"
    ],
    [
      "Exploits",
      "failed_event_construction"
    ],
    [
      "Reconnaissance",
      "failed_event_construction"
    ]
  ]
}


## 2. Construct the amended v0.20 suite

The same three-family denominator is retained. Known attack families may occur
in the warm-up, but the designated held-out family may not. Each warm-up class
requires at least 500 strictly earlier rows. The designated family must reach
at least 1% in both the first 500 and first 5,000 post-change flows.


In [4]:
suite = build_unsw_amended_event_suite(
    RAW,
    CACHE,
    EVENT_DIR,
    families=["DoS", "Exploits", "Reconnaissance"],
    source_max_rows=500_000,
    source_minimum_per_class=500,
    warmup_rows=20_000,
    post_change_rows=100_000,
    maximum_warmup_gap_hours=24.0,
    onset_windows=(500, 5_000),
    minimum_onset_prevalence=0.01,
    seed=11,
)
print(json.dumps(suite, indent=2))

status_by_family = {
    row["family"]: row["status"] for row in suite["outcomes"]
}
assert status_by_family == {
    "DoS": "failed_event_construction",
    "Exploits": "constructed",
    "Reconnaissance": "constructed",
}
assert suite["raw_dataset_sha256"] == EXPECTED_RAW_SHA256


Exploits: chunk 1, rows 250,000, elapsed 0.0 min
Exploits: chunk 10, rows 2,365,424, elapsed 0.3 min
Reconnaissance: chunk 1, rows 250,000, elapsed 0.0 min
Reconnaissance: chunk 10, rows 2,365,424, elapsed 0.3 min
{
  "protocol_id": "RAIDS-NIDS-v0.20-external-guard-amendment",
  "dataset": "NF-UNSW-NB15-v3",
  "amends_protocol": "RAIDS-NIDS-v0.19-external-guard-comparison",
  "amendment_trigger": "All three v0.19 episodes failed the all-benign warm-up rule",
  "prespecified_families": [
    "DoS",
    "Exploits",
    "Reconnaissance"
  ],
  "replacement_after_outcome": "prohibited",
  "raw_dataset_sha256": "4ebb97bd74412d566137d95a6fc3ffd8f374f1cf8cfe204d007848e7a668f9b5",
  "outcomes": [
    {
      "family": "DoS",
      "status": "failed_event_construction",
      "error_type": "AmendedEventConstructionError",
      "reason": "No eligible DoS occurrence satisfied the amended v0.20 construction rules; rejected={'insufficient_preceding_rows': 57, 'insufficient_post_rows': 255, 'emergi

## 3. Audit each constructed source and target


In [5]:
dataset_configs = {
    "DoS": (
        "configs/datasets/nf_unsw_nb15_v3_dos_v020_source.yaml",
        "configs/datasets/nf_unsw_nb15_v3_dos_v020_target.yaml",
    ),
    "Exploits": (
        "configs/datasets/nf_unsw_nb15_v3_exploits_v020_source.yaml",
        "configs/datasets/nf_unsw_nb15_v3_exploits_v020_target.yaml",
    ),
    "Reconnaissance": (
        "configs/datasets/"
        "nf_unsw_nb15_v3_reconnaissance_v020_source.yaml",
        "configs/datasets/"
        "nf_unsw_nb15_v3_reconnaissance_v020_target.yaml",
    ),
}
constructed = {
    row["family"]
    for row in suite["outcomes"]
    if row["status"] == "constructed"
}
audit_reports = {}
for family in sorted(constructed):
    source_cfg, target_cfg = dataset_configs[family]
    source_report = audit_dataset(
        source_cfg,
        Path("results/audits") / f"v020_{family}_source.json",
    )
    target_report = audit_dataset(
        target_cfg,
        Path("results/audits") / f"v020_{family}_target.json",
    )
    audit_reports[family] = {
        "source": source_report,
        "target": target_report,
    }
    print(
        family,
        "source rows=", source_report["rows_audited"],
        "target rows=", target_report["rows_audited"],
    )


Exploits source rows= 500000 target rows= 120000
Reconnaissance source rows= 500000 target rows= 120000


## 4. Review the amended episode manifests

Report additional target families rather than hiding them. No model or guard
is run in this section.


In [6]:
review_rows = []
for row in suite["outcomes"]:
    if row["status"] != "constructed":
        review_rows.append(
            {
                "family": row["family"],
                "status": row["status"],
                "event_time": None,
                "warmup_counts": None,
                "minimum_history": None,
                "post500": None,
                "post5000": None,
                "other_novel_target_families": None,
                "all_integrity_checks": None,
            }
        )
        continue
    manifest = json.loads(
        Path(row["manifest_path"]).read_text(encoding="utf-8")
    )
    checks = manifest["integrity_checks"]
    assert all(checks.values())
    assert manifest["raw_dataset_sha256"] == EXPECTED_RAW_SHA256
    review_rows.append(
        {
            "family": row["family"],
            "status": row["status"],
            "event_time": manifest["event_time"],
            "warmup_counts": json.dumps(
                manifest["warmup_family_counts"], sort_keys=True
            ),
            "minimum_history": manifest[
                "minimum_warmup_family_history"
            ],
            "post500": manifest["observed_onset_counts"]["500"],
            "post5000": manifest["observed_onset_counts"]["5000"],
            "other_novel_target_families": "|".join(
                manifest["other_novel_target_families"]
            )
            or "<none>",
            "all_integrity_checks": all(checks.values()),
        }
    )

episode_review = pd.DataFrame(review_rows)
print(episode_review.to_string(index=False))


        family                    status                 event_time                                      warmup_counts  minimum_history  post500  post5000 other_novel_target_families all_integrity_checks
           DoS failed_event_construction                       None                                               None              NaN      NaN       NaN                        None                 None
      Exploits               constructed 2015-02-18 01:04:57.146000                {"Backdoor": 2296, "Benign": 17704}            824.0      6.0     139.0                      <none>                 True
Reconnaissance               constructed 2015-02-18 01:06:32.190000 {"Backdoor": 2270, "Benign": 17724, "Exploits": 6}            857.0      6.0      55.0                      <none>                 True


## Stop here for the first review

Keep both flags below set to `False`. Send the complete v0.20 suite output and
the episode-review table before opening any model-based guard outcome.


In [8]:
# RUN_SEED11 = False
# RUN_FULL_MATRICES = False
RUN_SEED11 = True
RUN_FULL_MATRICES = False
print("RUN_SEED11 =", RUN_SEED11)
print("RUN_FULL_MATRICES =", RUN_FULL_MATRICES)


RUN_SEED11 = True
RUN_FULL_MATRICES = False


## 5. Authoritative seed 11

Activate this only after the construction artifacts have been reviewed. Every
eligible family uses one common score trace for MAD, ADWIN, and Page-Hinkley.


In [9]:
benchmark_configs = {
    "DoS": "configs/guard_benchmarks/v020_unsw_dos.yaml",
    "Exploits": "configs/guard_benchmarks/v020_unsw_exploits.yaml",
    "Reconnaissance": (
        "configs/guard_benchmarks/"
        "v020_unsw_reconnaissance.yaml"
    ),
}
seed11_summaries = {}
if RUN_SEED11:
    for family in ["DoS", "Exploits", "Reconnaissance"]:
        if family not in constructed:
            print(family, "skipped because event construction failed")
            continue
        summary = run_guard_benchmark(benchmark_configs[family])
        seed11_summaries[family] = summary
        print("\n", family)
        for result in summary["guard_results"]:
            print(
                result["detector"],
                result["guard_status"],
                result["post_change_detected"],
                result["detection_delay_windows"],
            )
else:
    print("Seed 11 is paused pending the construction review.")


DoS skipped because event construction failed

 Exploits
mad passed True 8
adwin passed True 91
page_hinkley passed True 0

 Reconnaissance
mad passed True 7
adwin passed True 90
page_hinkley passed True 7


In [10]:
import json
import pandas as pd

for family, summary in seed11_summaries.items():
    print("\n" + "=" * 80)
    print("FAMILY:", family)
    print("True-change window:", summary["true_change_window"])
    print("Window size:", summary["stream_window_size"])

    print("\nSELECTED GUARD RESULTS")
    selected = pd.DataFrame(summary["guard_results"])
    selected_columns = [
        "detector",
        "guard_status",
        "selected_parameter_name",
        "selected_parameter",
        "guard_safe_candidate_count",
        "calibration_trigger_count",
        "post_change_detected",
        "trigger_window",
        "detection_delay_windows",
        "trigger_normalized_score",
    ]
    print(selected[selected_columns].to_string(index=False))

    print("\nALL CANDIDATE AUDITS")
    candidate_audit = pd.read_csv(
        summary["files"]["candidate_audit"]
    )
    audit_columns = [
        "detector",
        "candidate_parameter",
        "candidate_value",
        "sensitivity_rank",
        "calibration_trigger_count",
        "calibration_trigger_windows",
        "guard_trigger_count",
        "guard_trigger_windows",
        "guard_safe",
    ]
    print(candidate_audit[audit_columns].to_string(index=False))

    print("\nCALIBRATION AND BOUNDARIES")
    print(json.dumps(summary["calibration"], indent=2))

    print("\nINTEGRITY CHECKS")
    print(json.dumps(summary["integrity_checks"], indent=2))

    score_trace = pd.read_csv(summary["files"]["score_trace"])

    trigger_windows = {
        int(row["trigger_window"])
        for row in summary["guard_results"]
        if row["trigger_window"] is not None
    }
    review_windows = set(range(35, 51)) | trigger_windows

    trace_columns = [
        "window",
        "phase",
        "shift_score",
        "normalized_shift_score",
        "predicted_unknown_rate",
        "novel_prevalence_posthoc",
        "labels_present_posthoc",
    ]

    print("\nCHANGE-POINT AND TRIGGER WINDOWS")
    print(
        score_trace.loc[
            score_trace["window"].isin(sorted(review_windows)),
            trace_columns,
        ].to_string(index=False)
    )


FAMILY: Exploits
True-change window: 40
Window size: 500

SELECTED GUARD RESULTS
    detector guard_status selected_parameter_name  selected_parameter  guard_safe_candidate_count  calibration_trigger_count  post_change_detected  trigger_window  detection_delay_windows  trigger_normalized_score
         mad       passed              multiplier                 3.0                           4                          0                  True              48                        8                       8.0
       adwin       passed                   delta                 0.1                           5                          0                  True             131                       91                       8.0
page_hinkley       passed               threshold                 5.0                           4                          2                  True              40                        0                       8.0

ALL CANDIDATE AUDITS
    detector candidate_parameter  candid

In [11]:
import joblib
import numpy as np
import pandas as pd

from raids_nids.data import load_dataset, align_feature_frames
from raids_nids.runner import _ordered_target

for family, summary in seed11_summaries.items():
    cfg = load_yaml(summary["files"]["resolved_config"])

    source = load_dataset(cfg["source_dataset"])
    target = load_dataset(cfg["target_dataset"])

    _, target_x, _ = align_feature_frames(
        source.features,
        target.features,
    )

    ordered_x, _, _ = _ordered_target(
        target_x,
        target.labels,
        target.time,
        summary["initial_known_classes"],
        cfg["stream"],
    )

    model = joblib.load(summary["files"]["model"])
    window_size = int(summary["stream_window_size"])

    ref_start = (
        int(summary["calibration"]["reference_start_window"])
        * window_size
    )
    ref_stop = (
        int(summary["calibration"]["reference_end_window"])
        * window_size
    )

    reference_embedding = model.embed(
        ordered_x.iloc[ref_start:ref_stop]
    )
    reference_mean = reference_embedding.mean(axis=0)
    reference_std = reference_embedding.std(axis=0) + 1e-6

    if getattr(model, "reducer", None) is None:
        try:
            feature_names = np.asarray(
                model.preprocessor.transformer.get_feature_names_out(),
                dtype=str,
            )
        except Exception:
            feature_names = np.asarray(
                [
                    f"embedding_{i}"
                    for i in range(reference_embedding.shape[1])
                ]
            )
    else:
        feature_names = np.asarray(
            [
                f"pca_component_{i}"
                for i in range(reference_embedding.shape[1])
            ]
        )

    trace = pd.read_csv(summary["files"]["score_trace"])
    prechange = trace[
        trace["window"] < summary["true_change_window"]
    ]

    review_windows = {
        int(trace.loc[trace["shift_score"].idxmax(), "window"]),
        int(
            prechange.loc[
                prechange["shift_score"].idxmax(),
                "window",
            ]
        ),
    }

    review_windows.update(
        int(row["trigger_window"])
        for row in summary["guard_results"]
        if row["trigger_window"] is not None
    )

    print("\n" + "=" * 80)
    print("FAMILY:", family)
    print(
        "Reference dimensions with std <= 1e-4:",
        int(np.sum(reference_std <= 1e-4)),
    )

    for window in sorted(review_windows):
        start = window * window_size
        stop = start + window_size
        embedding = model.embed(ordered_x.iloc[start:stop])

        standardized_change = (
            embedding.mean(axis=0) - reference_mean
        ) / reference_std

        squared = standardized_change**2
        contribution = 100.0 * squared / max(squared.sum(), 1e-12)
        recomputed_score = float(np.sqrt(np.mean(squared)))

        diagnostic = pd.DataFrame(
            {
                "feature": feature_names,
                "abs_standardized_change": np.abs(
                    standardized_change
                ),
                "contribution_percent": contribution,
                "reference_std": reference_std,
                "reference_mean": reference_mean,
                "window_mean": embedding.mean(axis=0),
            }
        ).sort_values(
            "contribution_percent",
            ascending=False,
        )

        saved_score = float(
            trace.loc[
                trace["window"] == window,
                "shift_score",
            ].iloc[0]
        )

        print(
            f"\nWindow {window}: "
            f"saved_score={saved_score:.6f}, "
            f"recomputed_score={recomputed_score:.6f}"
        )
        print(
            diagnostic.head(8).to_string(
                index=False,
                float_format=lambda value: f"{value:.6g}",
            )
        )


FAMILY: Exploits
Reference dimensions with std <= 1e-4: 1

Window 23: saved_score=310.640901, recomputed_score=310.640901
                         feature  abs_standardized_change  contribution_percent  reference_std  reference_mean  window_mean
         numeric__DNS_TTL_ANSWER                  2152.18               99.9996    0.000769226     -0.00248825      1.65302
         numeric__DNS_QUERY_TYPE                  4.30681           0.000400455     0.00396053     -0.00245564    0.0146016
   numeric__FTP_COMMAND_RET_CODE                 0.361473           2.82094e-06       0.933707        -0.07988      0.25763
       numeric__LONGEST_FLOW_PKT                 0.227374           1.11615e-06       0.979163      -0.0630606    -0.285697
         numeric__MAX_IP_PKT_LEN                 0.227374           1.11615e-06       0.979163      -0.0630606    -0.285697
         numeric__MIN_IP_PKT_LEN                 0.201229            8.7422e-07        1.33166       -0.102322     -0.37029
         

## 6. Remaining paired seeds

Activate only after reviewing the saved seed-11 score traces and candidate
audits. Do not change a candidate value.


In [ ]:
matrix_configs = {
    "DoS": "configs/matrices/v020_unsw_dos_guards.yaml",
    "Exploits": "configs/matrices/v020_unsw_exploits_guards.yaml",
    "Reconnaissance": (
        "configs/matrices/"
        "v020_unsw_reconnaissance_guards.yaml"
    ),
}
if RUN_FULL_MATRICES:
    assert RUN_SEED11, "Run and review seed 11 first"
    for family in ["DoS", "Exploits", "Reconnaissance"]:
        if family not in constructed:
            continue
        matrix = load_yaml(matrix_configs[family])
        base = load_yaml(matrix["base_benchmark"])
        for combination in itertools.product(*matrix["axes"].values()):
            override = {}
            for value in combination:
                override = deep_merge(override, value)
            summary = run_guard_benchmark(deep_merge(base, override))
            print(family, "seed", summary["seed"], "completed")
else:
    print("Full matrices are paused pending the seed-11 review.")


## 7. Aggregate after every eligible matrix finishes


In [ ]:
if RUN_FULL_MATRICES:
    aggregate_manifest = aggregate_guard_benchmarks(
        RESULTS_DIR,
        AGGREGATE_DIR,
    )
    print(json.dumps(aggregate_manifest, indent=2))
else:
    print("Aggregation is paused until the full matrices finish.")
